# Mangrove avoided EAD + Hurricane Melissa NDVI damage/recovery

This notebook combines two existing analyses in a **new standalone workflow**:

1. Mangrove-level avoided coastal-flood EAD attribution (`minimum` and `maximum` scenarios).
2. Mangrove NDVI change from Hurricane Melissa:
   - `2 months before` (2025-08-21 to 2025-10-21)
   - `2 months after` (2025-10-29 to 2025-12-29)
   - `months 3-4 after` (2025-12-30 to 2026-02-28)

Main output: per-mangrove table with avoided EAD + damage + recovery metrics, filtered to mangroves with positive avoided EAD attribution.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import matplotlib.pyplot as plt

pd.options.display.max_columns = 200
pd.options.display.float_format = lambda x: f"{x:,.4f}"


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "dphil_papers" / "dphil_paper_3").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing dphil_papers/dphil_paper_3")

ROOT = find_repo_root()
PAPER3 = ROOT / "dphil_papers" / "dphil_paper_3"

# Analysis controls
WRITE_OUTPUTS = True
FILTER_MODE = "either_positive"  # options: either_positive, minimum_positive, maximum_positive
MIN_BASELINE_NDVI = 0.20
REL_DAMAGE_THRESHOLD = -0.10  # damaged if (after2-before)/before < -0.10

# Inputs
mangroves_path = PAPER3 / "inputs" / "forces_of_nature_mangroves" / "mangroves.shp"
ndvi_before_path = PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_before_epsg3448_2025-08-21_to_2025-10-21.tif"
ndvi_after2_path = PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_2months_after_epsg3448_2025-10-29_to_2025-12-29.tif"
ndvi_after34_path = PAPER3 / "inputs" / "ndvi" / "HLS_masked_NDVI_months3to4_after_epsg3448_2025-12-30_to_2026-02-28.tif"

attribution_min_path = (
    PAPER3
    / "results/02_damage_estimates/coastal_flood_damages/results_coastal_minimum_scenario"
    / "damage_estimates/mangrove_attribution/mangrove_attribution_total_1000m.csv"
)
attribution_max_path = (
    PAPER3
    / "results/02_damage_estimates/coastal_flood_damages/results_coastal_maximum_scenario"
    / "damage_estimates/mangrove_attribution/mangrove_attribution_total_1000m.csv"
)

output_dir = PAPER3 / "results" / "threats" / "ndvi" / "mangrove_avoided_ead_damage_recovery"
output_dir.mkdir(parents=True, exist_ok=True)

for p in [mangroves_path, ndvi_before_path, ndvi_after2_path, ndvi_after34_path, attribution_min_path, attribution_max_path]:
    if not p.exists():
        raise FileNotFoundError(p)

print("ROOT:", ROOT)
print("Output directory:", output_dir)
print("FILTER_MODE:", FILTER_MODE)


In [ ]:
def compute_mangrove_ndvi_stats(
    mangrove_shp: Path,
    ndvi_before: Path,
    ndvi_after2: Path,
    ndvi_after34: Path,
    min_baseline_ndvi: float = 0.20,
    rel_damage_threshold: float = -0.10,
) -> pd.DataFrame:
    # Per-mangrove NDVI damage and recovery metrics from three aligned rasters.

    mangroves = gpd.read_file(mangrove_shp)
    if "ID" not in mangroves.columns:
        raise KeyError("Expected 'ID' column in mangrove shapefile")

    mangroves["Mangrove_ID"] = pd.to_numeric(mangroves["ID"], errors="coerce").astype("Int64")
    if mangroves["Mangrove_ID"].isna().any():
        raise ValueError("Some mangrove IDs could not be parsed as integers")

    records: list[dict] = []

    with rasterio.open(ndvi_before) as src_b, rasterio.open(ndvi_after2) as src_a2, rasterio.open(ndvi_after34) as src_a34:
        if src_b.crs != src_a2.crs or src_b.crs != src_a34.crs:
            raise ValueError("NDVI rasters must be in same CRS")
        if src_b.width != src_a2.width or src_b.height != src_a2.height:
            raise ValueError("Before and after2 rasters have different dimensions")
        if src_b.width != src_a34.width or src_b.height != src_a34.height:
            raise ValueError("Before and after34 rasters have different dimensions")
        if src_b.transform != src_a2.transform or src_b.transform != src_a34.transform:
            raise ValueError("NDVI rasters must share the same grid/transform")

        mangroves = mangroves.to_crs(src_b.crs)

        for row in mangroves.itertuples(index=False):
            geom = [row.geometry]

            try:
                b_arr, _ = mask(src_b, geom, crop=True, filled=True)
                a2_arr, _ = mask(src_a2, geom, crop=True, filled=True)
                a34_arr, _ = mask(src_a34, geom, crop=True, filled=True)
            except ValueError:
                # No overlap between polygon and raster grid
                records.append({
                    "Mangrove_ID": int(row.Mangrove_ID),
                    "Parish": getattr(row, "Parish", None),
                    "TYPE": getattr(row, "TYPE", None),
                    "area_ha": float(getattr(row, "HECTARES", np.nan)) if getattr(row, "HECTARES", None) is not None else np.nan,
                    "n_pixels": 0,
                    "n_eligible_baseline_ge_0_2": 0,
                })
                continue

            b = b_arr[0].astype(float)
            a2 = a2_arr[0].astype(float)
            a34 = a34_arr[0].astype(float)

            valid = np.isfinite(b) & np.isfinite(a2) & np.isfinite(a34)
            valid &= (b >= -1.0) & (b <= 1.0) & (a2 >= -1.0) & (a2 <= 1.0) & (a34 >= -1.0) & (a34 <= 1.0)

            if src_b.nodata is not None:
                valid &= b != src_b.nodata
            if src_a2.nodata is not None:
                valid &= a2 != src_a2.nodata
            if src_a34.nodata is not None:
                valid &= a34 != src_a34.nodata

            n_pixels = int(valid.sum())
            if n_pixels == 0:
                records.append({
                    "Mangrove_ID": int(row.Mangrove_ID),
                    "Parish": getattr(row, "Parish", None),
                    "TYPE": getattr(row, "TYPE", None),
                    "area_ha": float(getattr(row, "HECTARES", np.nan)) if getattr(row, "HECTARES", None) is not None else np.nan,
                    "n_pixels": 0,
                    "n_eligible_baseline_ge_0_2": 0,
                })
                continue

            bb = b[valid]
            aa2 = a2[valid]
            aa34 = a34[valid]

            damage_delta = aa2 - bb
            recovery_delta = aa34 - aa2
            gap_delta = aa34 - bb

            eligible = bb >= min_baseline_ndvi
            n_eligible = int(eligible.sum())

            rel_change_after2 = np.full_like(bb, np.nan)
            rel_change_after2[eligible] = (aa2[eligible] - bb[eligible]) / bb[eligible]
            damaged_gt10 = np.zeros_like(eligible, dtype=bool)
            damaged_gt10[eligible] = rel_change_after2[eligible] < rel_damage_threshold

            improved_after34_vs_after2 = aa34 > aa2
            recovered_to_before_by_after34 = aa34 >= bb

            rec = {
                "Mangrove_ID": int(row.Mangrove_ID),
                "Parish": getattr(row, "Parish", None),
                "TYPE": getattr(row, "TYPE", None),
                "area_ha": float(getattr(row, "HECTARES", np.nan)) if getattr(row, "HECTARES", None) is not None else np.nan,
                "n_pixels": n_pixels,
                "n_eligible_baseline_ge_0_2": n_eligible,
                "mean_ndvi_before": float(np.mean(bb)),
                "mean_ndvi_after2": float(np.mean(aa2)),
                "mean_ndvi_after34": float(np.mean(aa34)),
                "mean_damage_after2_minus_before": float(np.mean(damage_delta)),
                "mean_recovery_after34_minus_after2": float(np.mean(recovery_delta)),
                "mean_gap_after34_minus_before": float(np.mean(gap_delta)),
                "pct_improved_after34_vs_after2": float(100.0 * np.mean(improved_after34_vs_after2)),
                "pct_recovered_to_before_by_after34": float(100.0 * np.mean(recovered_to_before_by_after34)),
                "n_damaged_gt10pct_after2": int(damaged_gt10.sum()),
                "pct_damaged_gt10pct_after2_of_eligible": float(100.0 * damaged_gt10.sum() / n_eligible) if n_eligible > 0 else np.nan,
            }

            damaged_mask = damaged_gt10
            if damaged_mask.sum() > 0:
                drop = bb[damaged_mask] - aa2[damaged_mask]
                gain = aa34[damaged_mask] - aa2[damaged_mask]
                valid_rf = drop > 0
                if valid_rf.any():
                    rec["mean_recovery_fraction_of_drop_pct_damaged"] = float(np.mean(gain[valid_rf] / drop[valid_rf]) * 100.0)
                else:
                    rec["mean_recovery_fraction_of_drop_pct_damaged"] = np.nan
            else:
                rec["mean_recovery_fraction_of_drop_pct_damaged"] = np.nan

            records.append(rec)

    out = pd.DataFrame(records).sort_values("Mangrove_ID").reset_index(drop=True)
    return out


In [ ]:
ndvi_stats = compute_mangrove_ndvi_stats(
    mangrove_shp=mangroves_path,
    ndvi_before=ndvi_before_path,
    ndvi_after2=ndvi_after2_path,
    ndvi_after34=ndvi_after34_path,
    min_baseline_ndvi=MIN_BASELINE_NDVI,
    rel_damage_threshold=REL_DAMAGE_THRESHOLD,
)

attr_min = pd.read_csv(attribution_min_path).rename(
    columns={"Total_Avoided_EAD_USD_attributed": "avoided_ead_usd_min"}
)
attr_max = pd.read_csv(attribution_max_path).rename(
    columns={"Total_Avoided_EAD_USD_attributed": "avoided_ead_usd_max"}
)

for df in (attr_min, attr_max):
    df["Mangrove_ID"] = pd.to_numeric(df["Mangrove_ID"], errors="coerce").astype("Int64")

combined = (
    ndvi_stats
    .merge(attr_min[["Mangrove_ID", "avoided_ead_usd_min"]], on="Mangrove_ID", how="left")
    .merge(attr_max[["Mangrove_ID", "avoided_ead_usd_max"]], on="Mangrove_ID", how="left")
)
combined[["avoided_ead_usd_min", "avoided_ead_usd_max"]] = combined[["avoided_ead_usd_min", "avoided_ead_usd_max"]].fillna(0.0)

combined["positive_avoided_min"] = combined["avoided_ead_usd_min"] > 0
combined["positive_avoided_max"] = combined["avoided_ead_usd_max"] > 0
combined["positive_avoided_either"] = combined["positive_avoided_min"] | combined["positive_avoided_max"]

if FILTER_MODE == "minimum_positive":
    subset = combined[combined["positive_avoided_min"]].copy()
elif FILTER_MODE == "maximum_positive":
    subset = combined[combined["positive_avoided_max"]].copy()
elif FILTER_MODE == "either_positive":
    subset = combined[combined["positive_avoided_either"]].copy()
else:
    raise ValueError("FILTER_MODE must be one of: minimum_positive, maximum_positive, either_positive")

print("FoN mangroves total:", len(combined))
print("Mangroves matching filter:", len(subset))

show_cols = [
    "Mangrove_ID", "Parish", "TYPE",
    "avoided_ead_usd_min", "avoided_ead_usd_max",
    "mean_damage_after2_minus_before",
    "mean_recovery_after34_minus_after2",
    "mean_gap_after34_minus_before",
    "pct_damaged_gt10pct_after2_of_eligible",
    "pct_improved_after34_vs_after2",
    "pct_recovered_to_before_by_after34",
]
subset.sort_values("avoided_ead_usd_max", ascending=False)[show_cols].head(15)


In [ ]:
def weighted_mean(series: pd.Series, weights: pd.Series) -> float:
    m = series.notna() & weights.notna() & (weights > 0)
    if not m.any():
        return float("nan")
    return float(np.average(series[m], weights=weights[m]))

weights = subset["n_pixels"].astype(float)

summary_rows = [
    {"metric": "FoN mangroves (total)", "value": float(len(combined))},
    {"metric": f"Mangroves with positive avoided EAD ({FILTER_MODE})", "value": float(len(subset))},
    {"metric": "Total avoided EAD (minimum scenario, USD)", "value": float(subset["avoided_ead_usd_min"].sum())},
    {"metric": "Total avoided EAD (maximum scenario, USD)", "value": float(subset["avoided_ead_usd_max"].sum())},
    {"metric": "Mean damage (after2 - before)", "value": float(subset["mean_damage_after2_minus_before"].mean())},
    {"metric": "Weighted mean damage (after2 - before, by pixels)", "value": weighted_mean(subset["mean_damage_after2_minus_before"], weights)},
    {"metric": "Mean recovery in months 3-4 (after34 - after2)", "value": float(subset["mean_recovery_after34_minus_after2"].mean())},
    {"metric": "Weighted mean recovery in months 3-4 (after34 - after2, by pixels)", "value": weighted_mean(subset["mean_recovery_after34_minus_after2"], weights)},
    {"metric": "Mean remaining gap vs pre-event (after34 - before)", "value": float(subset["mean_gap_after34_minus_before"].mean())},
    {"metric": "Mean % damaged >10% at 2 months (eligible baseline only)", "value": float(subset["pct_damaged_gt10pct_after2_of_eligible"].mean())},
    {"metric": "Mean % improved in months 3-4 vs after2", "value": float(subset["pct_improved_after34_vs_after2"].mean())},
    {"metric": "Mean % recovered to/beyond before by months 3-4", "value": float(subset["pct_recovered_to_before_by_after34"].mean())},
]

summary = pd.DataFrame(summary_rows)
summary


In [ ]:
top_for_export = subset.sort_values("avoided_ead_usd_max", ascending=False).copy()

if WRITE_OUTPUTS:
    ndvi_stats.to_csv(output_dir / "mangrove_ndvi_damage_recovery_by_id.csv", index=False)
    combined.to_csv(output_dir / "mangrove_ndvi_damage_recovery_with_avoided_ead.csv", index=False)
    top_for_export.to_csv(output_dir / f"mangroves_positive_avoided_ead_{FILTER_MODE}.csv", index=False)
    summary.to_csv(output_dir / f"summary_mangroves_positive_avoided_ead_{FILTER_MODE}.csv", index=False)

    print("Saved:")
    print(" -", output_dir / "mangrove_ndvi_damage_recovery_by_id.csv")
    print(" -", output_dir / "mangrove_ndvi_damage_recovery_with_avoided_ead.csv")
    print(" -", output_dir / f"mangroves_positive_avoided_ead_{FILTER_MODE}.csv")
    print(" -", output_dir / f"summary_mangroves_positive_avoided_ead_{FILTER_MODE}.csv")
else:
    print("WRITE_OUTPUTS is False; no files written.")


In [ ]:
# Optional quick visuals for interpretation
plot_df = subset.copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(
    plot_df["avoided_ead_usd_max"],
    plot_df["pct_damaged_gt10pct_after2_of_eligible"],
    alpha=0.75,
    s=30,
)
axes[0].set_xlabel("Avoided EAD (maximum scenario, USD)")
axes[0].set_ylabel("Damaged pixels >10% at 2 months (%)")
axes[0].set_title("Attributed avoided EAD vs hurricane damage")

axes[1].scatter(
    plot_df["avoided_ead_usd_max"],
    plot_df["mean_recovery_after34_minus_after2"],
    alpha=0.75,
    s=30,
)
axes[1].axhline(0.0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("Avoided EAD (maximum scenario, USD)")
axes[1].set_ylabel("Mean NDVI recovery (after34 - after2)")
axes[1].set_title("Attributed avoided EAD vs months 3-4 recovery")

plt.tight_layout()
plt.show()


## Notes

- This notebook does **not** modify existing workflows; it only reads existing inputs/results and writes new output CSVs in `results/threats/ndvi/mangrove_avoided_ead_damage_recovery/`.
- `FILTER_MODE` controls which mangroves are included in the final subset:
  - `minimum_positive`: positive avoided EAD in minimum scenario
  - `maximum_positive`: positive avoided EAD in maximum scenario
  - `either_positive`: positive in either scenario
- Damage definition uses the same relative threshold logic as your prior analysis: `(after2 - before) / before < -0.10` with `before >= 0.20`.
